<a href="https://colab.research.google.com/github/rmadatt/AI-Newton-StudyGroupAfrica/blob/main/AI_Newton_Elastic_Collision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Newton StudyGroup Africa - Elastic Collision Improved

**Forked from Science-Discovery/AI-Newton**

Minimal runnable analogy to full AI-Newton system for low-resource learning.

Full AI-Newton needs Linux+CUDA+Maple+Rust. This version needs only Python.

Core idea: Experiment Base -> Observation Logs -> KB scan for zero-variance laws.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. Define Experiment Configuration

In [ ]:
elastic_collision_config = {
    "experiment_id": "elastic_collision_1d_001",
    "system_type": "two_body_collision",
    "parameters": {"time_step": 0.01, "total_duration": 3.0},
    "objects": [
        {"name": "Body_A", "mass": 2.0, "initial_position": 0.0, "initial_velocity": 4.0},
        {"name": "Body_B", "mass": 3.0, "initial_position": 8.0, "initial_velocity": -1.0}
    ]
}

def run_elastic_collision_experiment(config):
    obj_a, obj_b = config["objects"]
    m1, x1, v1 = obj_a["mass"], obj_a["initial_position"], obj_a["initial_velocity"]
    m2, x2, v2 = obj_b["mass"], obj_b["initial_position"], obj_b["initial_velocity"]
    dt = config["parameters"]["time_step"]
    duration = config["parameters"]["total_duration"]
    steps = int(duration / dt)
    logs = []
    collided = False
    t_coll = None
    for step in range(steps):
        t = step * dt
        if not collided and x1 >= x2:
            v1_f = ((m1 - m2) * v1 + 2 * m2 * v2) / (m1 + m2)
            v2_f = ((m2 - m1) * v2 + 2 * m1 * v1) / (m1 + m2)
            v1, v2 = v1_f, v2_f
            collided = True
            t_coll = t
        x1 += v1 * dt
        x2 += v2 * dt
        logs.append({"time": t, "body_a_position": x1, "body_a_velocity": v1, "body_b_position": x2, "body_b_velocity": v2, "m1": m1, "m2": m2})
    return logs, t_coll

dataset, t_coll = run_elastic_collision_experiment(elastic_collision_config)
print(f"Generated {len(dataset)} frames. Collision at t={t_coll}s")
print("Initial:", dataset[0])
print("Post:", dataset[-1])


## 2. Visualize for study group

In [ ]:
df = pd.DataFrame(dataset)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10,6), sharex=True)
ax1.plot(df["time"], df["body_a_position"], label="Body A 2kg", lw=2)
ax1.plot(df["time"], df["body_b_position"], label="Body B 3kg", lw=2)
ax1.axvline(t_coll, color="red", ls="--", label=f"Collision {t_coll}s")
ax1.set_ylabel("Position m")
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_title("Position vs Time")
ax2.plot(df["time"], df["body_a_velocity"], label="Body A vel", lw=2)
ax2.plot(df["time"], df["body_b_velocity"], label="Body B vel", lw=2)
ax2.axvline(t_coll, color="red", ls="--")
ax2.set_xlabel("Time s")
ax2.set_ylabel("Velocity m/s")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Discovery Method 1: Manual KB scan

In [ ]:
def discover_conservation_law(dataset):
    v1 = np.array([r["body_a_velocity"] for r in dataset])
    v2 = np.array([r["body_b_velocity"] for r in dataset])
    m1, m2 = 2.0, 3.0
    candidates = {
        "Total Velocity (v1+v2)": v1+v2,
        "Relative Velocity (v1-v2)": v1-v2,
        "Sum Squares (v1^2+v2^2)": v1**2+v2**2,
        "Linear Momentum (m1*v1+m2*v2)": m1*v1+m2*v2,
        "Kinetic Energy (0.5*m1*v1^2+0.5*m2*v2^2)": 0.5*m1*v1**2+0.5*m2*v2**2
    }
    print("Evaluating candidates...\n")
    conserved=[]
    for name, vals in candidates.items():
        var=np.var(vals)
        is_cons=np.isclose(vals[0], vals[-1], atol=1e-3) and var<1e-2
        status="CONSERVED LAW DISCOVERED! ✅" if is_cons else "Rejected"
        print(f"[{status}] {name} -> Var {var:.6f} | {vals[0]:.2f} -> {vals[-1]:.2f}")
        if is_cons: conserved.append(name)
    return conserved

discovered = discover_conservation_law(dataset)
print("\n=> Discovered:", discovered)


## 4. Discovery Method 2: Automated with gplearn symbolic regression
This is closer to real AI-Newton - we dont give candidates.

In [ ]:
# !pip install gplearn -q
from gplearn.genetic import SymbolicRegressor
v1 = np.array([r["body_a_velocity"] for r in dataset])
v2 = np.array([r["body_b_velocity"] for r in dataset])
m1 = np.array([r["m1"] for r in dataset])
m2 = np.array([r["m2"] for r in dataset])
X = np.column_stack([m1, m2, v1, v2])
y_mom = m1*v1 + m2*v2
y_en = 0.5*m1*v1**2 + 0.5*m2*v2**2
print(f"True momentum constant {y_mom[0]}, energy {y_en[0]}")
est = SymbolicRegressor(population_size=3000, generations=20, function_set=("add","sub","mul","div"), parsimony_coefficient=0.001, random_state=0, verbose=1)
est.fit(X, y_mom)
print("\n--- DISCOVERED FORMULA FOR MOMENTUM ---")
print(est._program)
print("X0=m1, X1=m2, X2=v1, X3=v2 -> should be X0*X2 + X1*X3")


## 5. Next steps for Study Group

- Add noise
- Try inelastic collision (only momentum conserved)
- Try 3 bodies
- Search for zero-variance expressions automatically

Push this as AI_Newton_Elastic_Collision_Improved.ipynb